# VedRishi AI - Fine-tuning on Kaggle
## QLoRA Training for Qwen 2.5 7B on Sacred Hindu Scriptures

This notebook fine-tunes Qwen 2.5 7B Instruct on Gita, Ramayana, and Mahabharata verses.

**Requirements:**
- GPU: NVIDIA T4 x2 (or better)
- RAM: 16GB+
- Internet: ON (for model download)

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl unsloth

## 2. Clone Repository & Load Dataset

In [ ]:
import os
import json
from datasets import Dataset

# Clone VedRishi repo
!git clone https://github.com/vedrishi-ai/vedrishi-ai.git
%cd vedrishi-ai

# Load instruction pairs
train_data = []
val_data = []

with open('training/outputs/vedrishi_train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            train_data.append(json.loads(line))

with open('training/outputs/vedrishi_val.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            val_data.append(json.loads(line))

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Dataset loaded: {train_dataset.num_rows} train, {val_dataset.num_rows} val")

## 3. Load Qwen 2.5 7B with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
MAX_SEQ_LENGTH = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("Model loaded successfully!")
print(f"Model type: {type(model)}")

## 4. Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA configured!")
model.print_trainable_parameters()

## 5. Format Dataset for Training

In [ ]:
ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = ALPACA_PROMPT.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Dataset formatted!")
print(f"Sample text: {train_dataset[0]['text'][:200]}...")

## 6. Configure Training Arguments

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    ),
)

print("Trainer configured!")
print(f"Total steps: {trainer.args.num_train_epochs * len(train_dataset) // (trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps)}")

## 7. Start Training

In [ ]:
import time

print("Starting training...")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

start_time = time.time()

# Train
trainer_stats = trainer.train()

end_time = time.time()
training_time = end_time - start_time

print(f"\nTraining completed in {training_time/60:.1f} minutes")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

## 8. Save Model

In [ ]:
# Save LoRA adapter
model.save_pretrained("vedrishi-lora")
tokenizer.save_pretrained("vedrishi-lora")

print("LoRA adapter saved to vedrishi-lora/")

# Save merged model (optional - for deployment)
# model.save_pretrained_merged("vedrishi-merged", tokenizer, save_method="merged_16bit")
# print("Merged model saved to vedrishi-merged/")

## 9. Test Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test queries
test_queries = [
    "कर्मयोग क्या है?",
    "गीता में धर्म क्या है?",
    "मुझे जीवन में शांति चाहिए",
    "रामायण में हनुमान जी की भक्ति के बारे में बताइए",
    "महाभारत की सबसे बड़ी शिक्षा क्या है?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    
    inputs = tokenizer(query, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Response: {response[:300]}...")

## 10. Upload to HuggingFace Hub

In [ ]:
# Optional: Upload to HuggingFace Hub
# from huggingface_hub import login
# login()  # Enter your HuggingFace token

# model.push_to_hub("vedrishi-ai/vedrishi-lora", token=True)
# tokenizer.push_to_hub("vedrishi-ai/vedrishi-lora", token=True)
# print("Model uploaded to HuggingFace Hub!")

print("Training complete! Model saved locally.")